# Feature Engineering & Linear Algebra in ML
### Simple runnable examples for the classroom

Run each cell top to bottom (press **Shift + Enter**). Every cell prints a
**before → after** result so you can see exactly what each idea does on real data.

**Part 1 — Feature Engineering** (Topic 5)
**Part 2 — Linear Algebra** (Topic 6)


In [1]:
# One-time setup: import the tools we need
import numpy as np
import pandas as pd
print("Ready! pandas", pd.__version__, "| numpy", np.__version__)

Ready! pandas 3.0.5 | numpy 2.4.4


---
# PART 1 — Feature Engineering
Turning raw, messy data into a clean table of numbers a model can learn from.

## Our messy dataset
Real data is messy. Notice the problems: **text** in the `city` column, a **missing**
age (`NaN`), a huge **outlier** in `income`, and columns on very **different scales**.

In [2]:
df = pd.DataFrame({
    "age":    [25, 41, np.nan, 33],      # one value is MISSING
    "income": [3200, 5100, 2800, 999999],# one value is a huge OUTLIER
    "city":   ["Fergana", "Tashkent", "Namangan", "Fergana"],  # TEXT, not numbers
    "size":   ["low", "high", "medium", "low"],  # ORDERED categories
    "churn":  [0, 1, 0, 0],              # the TARGET we predict (y)
})
df

,age,income,city,size,churn
0,25.0,3200,Fergana,low,0
1,41.0,5100,Tashkent,high,1
2,NaN,2800,Namangan,medium,0
3,33.0,999999,Fergana,low,0


### The vocabulary
- **Feature matrix `X`** = the input columns
- **Sample** = one row (one customer) &nbsp; | &nbsp; **Feature** = one column (one property)
- **Target `y`** = what we want to predict

In [3]:
X = df[["age", "income", "city", "size"]]   # features
y = df["churn"]                             # target
print("X shape =", X.shape, " -> (samples, features)")
print("One SAMPLE (row 0):", X.iloc[0].to_dict())
print("One FEATURE (age): ", X["age"].tolist())
print("TARGET y:          ", y.tolist())

X shape = (4, 4)  -> (samples, features)
One SAMPLE (row 0): {'age': 25.0, 'income': 3200, 'city': 'Fergana', 'size': 'low'}
One FEATURE (age):  [25.0, 41.0, nan, 33.0]
TARGET y:           [0, 1, 0, 0]


## Technique 1 — Encoding: turn text into numbers
A model cannot do math on the word "Fergana". We turn text into numbers.

**One-hot** for categories with **no order** (cities). **Ordinal** for categories with a **real order** (low < medium < high).

In [4]:
# ONE-HOT: one 0/1 column per city (no fake ranking)
one_hot = pd.get_dummies(df["city"], prefix="city").astype(int)
print("ONE-HOT (for city - no order):")
print(one_hot)

# ORDINAL: keep the real order low < medium < high
order = {"low": 0, "medium": 1, "high": 2}
print("\nORDINAL (for size - real order):")
print(df["size"].map(order).tolist(), "  <- order preserved (0 < 1 < 2)")

ONE-HOT (for city - no order):
   city_Fergana  city_Namangan  city_Tashkent
0             1              0              0
1             0              0              1
2             0              1              0
3             1              0              0

ORDINAL (for size - real order):
[0, 2, 1, 0]   <- order preserved (0 < 1 < 2)


**The trap:** never give cities numbers like 1, 2, 3 — that tells the model
`Namangan > Fergana`, which is nonsense. Use one-hot for unordered text.

## Technique 2 — Binning: turn a number into a range
Sometimes the *band* matters more than the exact value.

In [5]:
age_filled = df["age"].fillna(df["age"].median())  # fill gap first
age_band = pd.cut(age_filled, bins=[0, 30, 45, 100],
                  labels=["young", "middle", "senior"])
pd.DataFrame({"age": age_filled, "age_band": age_band})

,age,age_band
0,25.0,young
1,41.0,middle
2,33.0,middle
3,33.0,middle


## Technique 3 — Imputing: fill the missing gaps
A gap (`NaN`) breaks the model. We fill it. **Median** is safe when there are outliers.

In [6]:
from sklearn.impute import SimpleImputer

print("age before:", df["age"].tolist())
median_filled = SimpleImputer(strategy="median").fit_transform(df[["age"]]).ravel()
print("age after :", median_filled, "  <- NaN became the median (33)")

age before: [25.0, 41.0, nan, 33.0]
age after : [25. 41. 33. 33.]   <- NaN became the median (33)


## Technique 4 — Outliers: cap or remove
An outlier is a value far from the rest (income = 999999). The **IQR rule** finds it.

In [7]:
inc = df["income"]
q1, q3 = inc.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
print(f"normal range: {low:.0f} to {high:.0f}")

print("Option A - CAP  :", np.clip(inc, low, high).tolist(), " (keeps all rows)")
print("Option B - REMOVE:", inc[(inc >= low) & (inc <= high)].tolist(), " (drops the outlier row)")

normal range: -372987 to 629912
Option A - CAP  : [3200.0, 5100.0, 2800.0, 629911.875]  (keeps all rows)
Option B - REMOVE: [3200, 5100, 2800]  (drops the outlier row)


## Technique 5 — Scaling: put columns on the same range
Without scaling, `income` (thousands) dominates `age` (tens) just because its numbers
are bigger. **Standardization** fixes this: mean 0, std 1.

In [8]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

ages = np.array([[25], [40], [55]])
print("original age :", ages.ravel().tolist())
print("standardized :", StandardScaler().fit_transform(ages).ravel().round(2).tolist(),
      "  <- mean 0, std 1")
print("min-max [0,1]:", MinMaxScaler().fit_transform(ages).ravel().round(2).tolist(),
      "  <- squeezed into 0..1")

original age : [25, 40, 55]
standardized : [-1.22, 0.0, 1.22]   <- mean 0, std 1
min-max [0,1]: [0.0, 0.5, 1.0]   <- squeezed into 0..1


## The golden rule — avoid DATA LEAKAGE
Learn the scaling numbers from the **training data only**, then apply to the test data.
If you use the test data to learn them, information "leaks" and your score becomes a lie.

In [9]:
from sklearn.preprocessing import StandardScaler

train = np.array([[10], [20], [30], [40]])   # training split
test  = np.array([[100], [5]])               # test split

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train)   # LEARN + apply on train
test_scaled  = scaler.transform(test)        # only APPLY on test (no re-learning)

print("train scaled:", train_scaled.ravel().round(2).tolist())
print("test  scaled:", test_scaled.ravel().round(2).tolist(), " <- used TRAIN's numbers = no leakage")

train scaled: [-1.34, -0.45, 0.45, 1.34]
test  scaled: [6.71, -1.79]  <- used TRAIN's numbers = no leakage


## Putting it together — a clean pipeline
A `Pipeline` does the steps in the right order and **prevents leakage automatically**.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# a tiny numeric example: fill gaps -> scale -> train, all in one safe object
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("model",  LogisticRegression()),
])
Xn = df[["age", "income"]]        # numeric features (age has a NaN)
pipe.fit(Xn, y)                   # every step fits on training data only
print("Pipeline trained safely. Predictions:", pipe.predict(Xn).tolist())

Pipeline trained safely. Predictions: [0, 1, 0, 0]


---
# PART 2 — Linear Algebra
The mathematics that operates on the clean matrix `X`.

## The three building blocks
- **Scalar** = one number &nbsp; | &nbsp; **Vector** = a 1-D list &nbsp; | &nbsp; **Matrix** = a 2-D grid

In [11]:
scalar = 5
vector = np.array([2, 5, 3])            # one sample
matrix = np.array([[1, 2, 3], [4, 5, 6]])  # a dataset / an image
print("scalar:", scalar)
print("vector:", vector, "shape", vector.shape)
print("matrix:\n", matrix, "\nshape", matrix.shape)

scalar: 5
vector: [2 5 3] shape (3,)
matrix:
 [[1 2 3]
 [4 5 6]] 
shape (2, 3)


## Vectors are POINTS, matrices are TRANSFORMATIONS
A vector is a location. A matrix is a rule that **moves** that point.

In [12]:
point = np.array([4, 3])          # a vector = a point in space
A = np.array([[1, 0.5],           # a matrix = a transformation
              [0.3, 1]])
print("before:", point)
print("after :", A @ point, "  <- the matrix moved the point")

before: [4 3]
after : [5.5 4.2]   <- the matrix moved the point


## Vector operations

In [13]:
a = np.array([3, 1]); b = np.array([1, 2])
print("addition   a + b =", (a + b).tolist(), " (combine)")
print("scalar mul 2 * a =", (2 * a).tolist(), " (stretch, same direction)")

addition   a + b = [4, 3]  (combine)
scalar mul 2 * a = [6, 2]  (stretch, same direction)


## The dot product — how a model predicts
Multiply matching entries, then add them into **one number**. This is exactly how a
model turns features into a prediction: `prediction = w . x`.

In [14]:
w = np.array([0.5, 2.0, 1.0])    # weights (importance of each feature)
x = np.array([10,  3,   4])      # one sample's feature values
print("step by step: 0.5*10 + 2.0*3 + 1.0*4 =", 0.5*10 + 2.0*3 + 1.0*4)
print("w @ x =", w @ x, "  <- the prediction")

# predict a WHOLE dataset at once with one matrix multiply
X = np.array([[10, 3, 4],
              [ 2, 1, 8],
              [ 5, 5, 5]])
print("\nAll rows at once  X @ w =", (X @ w).tolist())

step by step: 0.5*10 + 2.0*3 + 1.0*4 = 15.0
w @ x = 15.0   <- the prediction

All rows at once  X @ w = [15.0, 11.0, 17.5]


## Matrix operations

In [15]:
M = np.array([[1, 2], [3, 4]])
print("matrix x matrix  M @ M =\n", (M @ M))       # compose / batch
print("\ntranspose  M.T =\n", M.T)                 # flip rows <-> columns
print("\ninverse  inv(M) =\n", np.linalg.inv(M).round(2))  # the 'undo'
print("\ncheck  M @ inv(M) =\n", (M @ np.linalg.inv(M)).round(2), " (identity = undo works)")

matrix x matrix  M @ M =
 [[ 7 10]
 [15 22]]

transpose  M.T =
 [[1 3]
 [2 4]]

inverse  inv(M) =
 [[-2.   1. ]
 [ 1.5 -0.5]]

check  M @ inv(M) =
 [[1. 0.]
 [0. 1.]]  (identity = undo works)


## Dependent vs independent features
If one column is just a copy of another (height in cm vs m), it adds **no new information**
and breaks the math. `matrix_rank` tells us how many *truly different* directions we have.

In [16]:
dependent   = np.array([[2, 1], [4, 2]])   # row2 = 2 * row1  (redundant)
independent = np.array([[2, 1], [1, 2]])   # genuinely different
print("dependent   rank =", np.linalg.matrix_rank(dependent),  " (only 1 real direction)")
print("independent rank =", np.linalg.matrix_rank(independent), " (2 directions = full info)")

dependent   rank = 1  (only 1 real direction)
independent rank = 2  (2 directions = full info)


## The payoff — linear regression is just linear algebra
The best-fit line comes straight from matrix operations: **β = (Xᵀ X)⁻¹ Xᵀ y**.

In [17]:
# data that follows y = 2*x  (plus an intercept of 0)
x_data = np.array([1, 2, 3, 4, 5.0])
y_data = np.array([2, 4, 6, 8, 10.0])

Xd = np.c_[np.ones_like(x_data), x_data]         # add a column of 1s for the intercept
beta = np.linalg.inv(Xd.T @ Xd) @ Xd.T @ y_data  # the famous formula
print("intercept, slope =", beta.round(2), "  <- it found slope 2, exactly right")

intercept, slope = [0. 2.]   <- it found slope 2, exactly right


---
## Summary
**Part 1:** encode text, bin numbers, impute gaps, cap outliers, scale columns —
and always **fit on train only** to avoid leakage.

**Part 2:** data is a **matrix**, a prediction is a **dot product**, and learning is
**solving matrix equations**.

Feature engineering builds the clean matrix `X`; linear algebra does the math on it.